In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pygmo as pg
from matplotlib.collections import LineCollection
import matplotlib.ticker as ticker


In [ ]:
result_dir = 'your result dir'

In [ ]:
def frange(start, stop, step):
    while start < stop:
        yield round(start, 10) 
        start += step

In [ ]:
# List to store the results
cup_results = []
lr_list = [1e-3, 1e-2]

for lr in lr_list:
# Iterate over cup_gamma values from 0.01 to 0.99
    for gamma in [round(x, 2) for x in list(frange(0.01, 1.00, 0.01))]:
        # Initialize lists to store results for each seed
        retain_list = []
        correctness_list = []
        test_list = []
        confidence_list = []

        # Loop through seed values from 1 to 5
        for seed in range(1, 5):
            # Generate the path for the result file based on gamma and seed
            file_path = os.path.join(result_dir, f'GA_cup_cifar10_{seed}_{gamma}_lr_{lr}.txt')
            
            if os.path.exists(file_path):
                try:
                    # Open the file, replace single quotes with double quotes, and evaluate the content
                    with open(file_path, 'r') as f:
                        data_str = f.read().replace("'", '"')  # Convert single quotes to double quotes
                        data = eval(data_str)  # Evaluate the string to convert it into a Python dictionary

                    # Extract necessary values and append to corresponding lists
                    retain_list.append(data['accuracy']['retain'])
                    correctness_list.append(100-data['accuracy']['forget'])  # Correctness as percentage
                    test_list.append(data['accuracy']['test'])
                    confidence_list.append(data['SVC_MIA_forget_efficacy']['confidence']*100)
                
                except SyntaxError as e:
                    print(f"Error parsing data from {file_path}: {e}")
            else:
                print(f"File not found: {file_path}")

        # Compute the mean and standard deviation for each metric if data is available
        if retain_list:  # Ensure that data was found for this gamma
            cup_results.append({
                'cup_gamma': gamma,
                'retain_mean': np.mean(retain_list),
                'retain_std': np.std(retain_list),
                'correctness_mean': np.mean(correctness_list),
                'correctness_std': np.std(correctness_list),
                'test_mean': np.mean(test_list),
                'test_std': np.std(test_list),
                'confidence_mean': np.mean(confidence_list),
                'confidence_std': np.std(confidence_list)
            })

# Convert the results to a pandas DataFrame
cup_df = pd.DataFrame(cup_results)



### Hypervolume Indicator

In [ ]:
reference_point = [0, 0]
cup_front = cup_df[['retain_mean', 'correctness_mean']].values


In [ ]:
cup_hv = pg.hypervolume(-cup_front)
cup_hv_value = cup_hv.compute(reference_point)
print(cup_hv_value)

### Visualization

In [ ]:


# Increase figure size
plt.figure(figsize=(10, 6))

# Retrain
plt.scatter(100.0, 100.0, c='red',marker='*', s=500, edgecolor='none', gamma=0.85, label='Retrain')

# cup-MU
scatter = plt.scatter(cup_df['retain_mean'], cup_df['correctness_mean'], c=cup_df['cup_gamma'], cmap='viridis', s=150, edgecolor='none', gamma=0.85, label='cup-MU')

# Add gridlines
plt.grid(True, which='both', linestyle='--', linewidth=0.5, gamma=0.7)

# Add a color bar to the plot
cbar = plt.colorbar(scatter)
cbar.set_label(r'$\gamma$', fontsize=25, labelpad=5)  # Set the font size for the color bar label
cbar.ax.tick_params(labelsize=20)  # Set the font size for the color bar ticks

# Set color bar ticks manually from 0.0 to 1.0
cbar.set_ticks([0.01, 0.3, 0.6, 0.9])
cbar.set_ticklabels(['0.01', '0.3', '0.6','0.9' ])  # Ensure labels show the full range

# Set the tick size and format for the x-axis and y-axis
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)


# Add axis labels with larger and bold fonts
plt.xlabel('Remaining Accuracy (%)', fontsize=22)
plt.ylabel('Unlearning Accuracy (%)', fontsize=22)

# Set the title with larger and bold font

# Display the plot
plt.legend(fontsize=22, loc='lower left')
plt.tight_layout()  # Automatically adjust subplot parameters to give some padding
plt.savefig('./figures/ParetoFront_cifar10.png')
plt.savefig('./figures/ParetoFront_cifar10.pdf')
plt.show()